# GF(8) Toric Code Champion Search — Local (Apple Silicon)

In [ ]:
import os
os.environ.setdefault("PYOPENCL_CTX", "0")

from precompute import init_opencl, bounding_cube_points
from kernel import DistanceOracle
from champion_search import find_latest_results, load_results, check_set, global_batched_bfs, idx
from pathlib import Path
from datetime import datetime

ctx, queue_cl, M_buf, _ = init_opencl()
oracle = DistanceOracle(ctx, queue_cl, M_buf)
lattice = bounding_cube_points()

## Best known bounds for [49, k] codes over GF(8)

In [ ]:
from codetables import bounds_for_n
import pandas as pd

targets = bounds_for_n(49)
display(pd.DataFrame(
    [(k, d) for k, d in sorted(targets.items()) if k <= 15],
    columns=["k", "best_known_d"]
))

In [ ]:
results_file = find_latest_results()
if results_file is None:
    results_file = Path(f"champions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    print(f"Starting fresh: {results_file}")
else:
    print(f"Resuming from: {results_file}  ({len(load_results(results_file))} records)")

## Part A — Structured geometric candidates

In [ ]:
already_done = (
    {rec["name"] for rec in load_results(results_file) if "name" in rec}
    if results_file.exists() else set()
)

circle_8 = sorted([idx(0,1), idx(0,6), idx(1,0), idx(2,2), idx(2,5), idx(5,2), idx(5,5), idx(6,0)])
check_set("circle_conic_k8", circle_8, oracle, lattice, results_file, already_done=already_done)

parabola_7 = sorted([idx(t, (t*t) % 7) for t in range(7)])
check_set("parabola_conic_k7", parabola_7, oracle, lattice, results_file, already_done=already_done)

## Part B — Global BFS guided by codetables.de bounds

The table gives the best known d for *any* linear [49, k] code over GF(8).
Toric codes are a restricted subclass and typically fall 0–3 points short of those bounds.
`margin` subtracts that gap so the BFS doesn't prune everything on the first level.
Increase it if level k=2 shows 0 survivors; decrease it to search more aggressively.

In [ ]:
import time

margin = 3  # raise if BFS dies at k=2 (target too high); lower to search harder
adjusted = {k: max(d - margin, 1) for k, d in targets.items()}
print("Adjusted per-k targets:", {k: adjusted[k] for k in sorted(adjusted) if k <= 10})

t0 = time.perf_counter()
global_batched_bfs(
    target_distance=adjusted,
    oracles=oracle,       # single oracle; function accepts DistanceOracle or list
    lattice=lattice,
    results_file=results_file,
    max_k=10,
    resume=True,
)
print(f"\nTotal BFS time: {time.perf_counter() - t0:.1f}s")

## Results vs. best known bounds

In [ ]:
records = [r for r in load_results(results_file) if r.get("type") != "level_complete"]
df = pd.DataFrame(records)[["name", "k", "min_distance", "indices"]]
df["best_known_d"] = df["k"].map(targets)
df["gap"] = df["best_known_d"] - df["min_distance"]
df["new_record"] = df["gap"] < 0
df.sort_values(["k", "min_distance"], ascending=[True, False], inplace=True)
df.reset_index(drop=True, inplace=True)
display(df)